# How many things is this actually measuring?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

The project rests on a sentence: **a player must clear a floor on every
requirement**. Independent hurdles, each one a different demand, and only 6% of
players clear all of them.

That sentence has a load-bearing word in it, and it is not the count. It is
"independent", which nobody had checked. Checking it cost the project one of its
requirements.

## 1. Question

Ten requirements go into the gate. How many separate things do they actually
test?

## 2. Intuition

Here is the arithmetic that should have bothered me from the beginning.

Each requirement rejects the bottom 40% of players. If they were independent,
the share clearing all ten would be $0.6^{10}$, which is **0.60%**, or about
thirty players out of five and a half thousand. The real figure is six per cent,
ten times higher.

That gap is not a mistake. It is correlation, and it is what you would expect:
good players tend to be good at several things at once. But it means the gate is
nothing like ten hurdles. It is a smaller number of hurdles wearing ten labels,
and until you know how many, "must have all of them" is a claim about the label
count rather than about football.

There is a second thing worth finding out. If two requirements measure the same
underlying quality, the composite counts that quality twice, and a player strong
in it is rewarded twice for one virtue.

## 3. Math

Three measurements, from weakest to most useful.

**The correlation matrix** $C$, on the standardised career profile the gate
actually sees. One entry per pair.

**Eigenvalues.** $C$ has as many as there are requirements and they sum to that
number. If the requirements were independent every eigenvalue would be 1.
Concentration in the first few means the variation lives in fewer dimensions
than there are columns. The **participation ratio** turns that into one number:

$$k_{\text{eff}} = \frac{\left(\sum_i \lambda_i\right)^2}{\sum_i \lambda_i^2}$$

**Equivalent independent requirements.** The most directly interpretable of the
three. If the observed pass rate is $p$ and each floor passes 60%, then the gate
behaves like $k$ independent floors where $0.6^k = p$:

$$k = \frac{\ln p}{\ln 0.6}$$

That last one answers the question in the units the claim was made in.

## 4. Code

### The matrix

In [1]:
import numpy as np
import pandas as pd

from gambeta import needs

ranking = pd.read_parquet("../data/sample/ranking.parquet")
keys = [r.key for r in needs.OUTFIELD]
labels = {r.key: r.label for r in needs.OUTFIELD}

C = ranking[keys].corr()
C.rename(index=labels, columns=labels).style.format("{:.2f}").background_gradient(
    cmap="RdBu_r", vmin=-1, vmax=1, axis=None
)

,Scores goals,Creates goals,Finishes clinically,Generates threat,Carries his team,Is available,Sees matches out,Sustains it,Has no bad seasons,Does not cost his team
Scores goals,1.00,0.57,0.57,0.94,0.92,-0.15,0.10,0.07,0.81,-0.12
Creates goals,0.57,1.00,0.30,0.67,0.52,-0.11,-0.01,0.12,0.61,-0.06
Finishes clinically,0.57,0.30,1.00,0.47,0.57,-0.07,0.15,0.08,0.52,-0.19
Generates threat,0.94,0.67,0.47,1.00,0.90,-0.17,0.08,0.05,0.79,-0.15
Carries his team,0.92,0.52,0.57,0.90,1.00,0.00,0.22,0.06,0.81,-0.12
Is available,-0.15,-0.11,-0.07,-0.17,0.00,1.00,0.54,0.28,0.25,0.30
Sees matches out,0.10,-0.01,0.15,0.08,0.22,0.54,1.00,0.19,0.37,-0.03
Sustains it,0.07,0.12,0.08,0.05,0.06,0.28,0.19,1.00,0.11,0.06
Has no bad seasons,0.81,0.61,0.52,0.79,0.81,0.25,0.37,0.11,1.00,0.17
Does not cost his team,-0.12,-0.06,-0.19,-0.15,-0.12,0.30,-0.03,0.06,0.17,1.00


Read the top-left block first. `scoring`, `threat`, `team_share` and
`consistency` sit between 0.79 and 0.94 with each other. That is not four
separate requirements, it is one attacking quality measured four ways.

Then the pair lower down: `availability` and `reliability` at **0.7**. Being on
the pitch and being picked to start are, as measured here, largely the same fact
about a player.

In [2]:
# Upper triangle only, so each pair is named once.
upper = C.where(~np.tril(np.ones(C.shape, dtype=bool)))
pairs = (
    upper.melt(ignore_index=False, var_name="other", value_name="r")
    .dropna()
    .set_index("other", append=True)["r"]
    .sort_values(key=abs, ascending=False)
)
print("the six strongest relationships between supposedly separate requirements:\n")
for (a, b), v in pairs.head(6).items():
    print(f"  {labels[a]:<24} {labels[b]:<24} {v:+.4f}")

the six strongest relationships between supposedly separate requirements:

  Scores goals             Generates threat         +0.9439
  Scores goals             Carries his team         +0.9188
  Generates threat         Carries his team         +0.9023
  Carries his team         Has no bad seasons       +0.8145
  Scores goals             Has no bad seasons       +0.8111
  Generates threat         Has no bad seasons       +0.7925


**The largest of those used to be 0.9999, and it is why this chapter exists.**

There was an eleventh requirement, `above_team`, meant to separate a player from
the side around him: score 20 for a title winner and it should count for less
than 20 for a relegation side. It was built as the residual of scoring after
regressing on the club's ClubElo rating.

It correlated **0.9999** with `scoring`, and the smallest eigenvalue of the
matrix was zero, which is linear algebra saying one column is a copy of another.

Two things were wrong. The ClubElo adapter had been filtering to England since
Phase 1, so four leagues in five had no rating at all and were quietly given the
average. And once that was fixed, club strength turned out to explain **0.8%** of
who scores inside a league-season. The residual of something barely explained is
the thing back again.

No rebuild rescued it. Every other requirement was measured against club
strength and the best was `creation` at 1.1%. Attacking output is individual.

So the definition lost a requirement and went from eleven to ten. The
composite had been counting one quality twice, and removing it moved the answer:
Mbappé left the top five, Kane and Henry moved up.

**It survives on the goalkeeper list**, where the same construction explains
**45%** of goals conceded. A keeper's goals-against is mostly his defence, and
the residual is the part that is his. Fixing the England-only bug turned that
board over: Cañizares now leads it.

### How many hurdles is the gate really?

In [3]:
eigenvalues = np.linalg.eigvalsh(C.to_numpy())[::-1]
observed = ranking["qualified"].mean()

print("eigenvalues (they sum to 11; all ones would mean independence):")
print("  " + "  ".join(f"{v:.2f}" for v in eigenvalues))
print()
print(f"participation ratio          {eigenvalues.sum() ** 2 / (eigenvalues**2).sum():.2f}")
print(f"components for 90% variance  {int(np.searchsorted(np.cumsum(eigenvalues) / 11, 0.90) + 1)}")
print()
print(f"pass rate if independent     {0.6**11:.4%}")
print(f"pass rate observed           {observed:.4%}")
print(f"equivalent independent gates {np.log(observed) / np.log(0.6):.2f}")

eigenvalues (they sum to 11; all ones would mean independence):
  4.47  1.88  1.08  0.91  0.65  0.48  0.34  0.09  0.06  0.04

participation ratio          3.80
components for 90% variance  8

pass rate if independent     0.3628%
pass rate observed           8.0792%
equivalent independent gates 4.93


**Ten requirements behave like about five and a half.**

The three measures disagree with each other, and the disagreement is honest
rather than a problem. The participation ratio weights the largest eigenvalue
heavily and gives about three and a half. Ninety per cent of the variance needs
five components. The pass-rate calculation gives 5.4. They ask slightly
different questions, and all three land between three and six, which is the
answer: **the gate tests roughly half as many things as it says it does.**

### Does correlation actually explain the gap?

The claim above is that correlation, and nothing else, accounts for six per cent
rather than 0.6%. That is testable. Draw fake players with exactly this
correlation structure and gate them the same way.

In [4]:
rng = np.random.default_rng(20260810)
draws = rng.multivariate_normal(np.zeros(len(keys)), C.to_numpy(), size=200_000)

floors = np.percentile(draws, 40, axis=0)
simulated = (draws >= floors).all(axis=1).mean()

print(f"independent requirements     {0.6**11:.4%}")
print(f"simulated, this correlation  {simulated:.4%}")
print(f"observed in the real data    {observed:.4%}")

independent requirements     0.3628%
simulated, this correlation  6.3645%
observed in the real data    8.0792%


Close enough to call it. The correlation structure alone reproduces the
qualifier rate, so nothing else needs explaining: the gate is not unexpectedly
generous, it is exactly as generous as ten overlapping tests should be.

What is left over is worth noticing too. The simulation assumes joint normality
and the real requirements are skewed, which is why the two do not match exactly.

## 5. Assumptions

1. **Correlation captures the dependence.** Pearson sees linear relationships.
   Two requirements could be strongly dependent in a curved way and show a
   correlation near zero.
2. **The career profile is the right place to measure.** These are pooled,
   standardised career values, which is what the gate sees. The season-level
   picture is similar but not identical.
3. **The simulation's normality is wrong and useful anyway.** It is there to show
   that correlation is sufficient to explain the pass rate, not to model the
   distribution faithfully.

## 6. How it breaks

**Effective dimensionality is not a verdict on any single requirement.**

It would be easy to read "five and a half" as "delete five requirements", and
that does not follow. A requirement can be highly correlated with others and
still be the one doing the eliminating at the margin. `discipline` correlates
weakly with everything, and it rejects more players on its own than any other.
Correlation describes the population; the gate acts on individuals.

The one case where the reading was safe was a correlation of 0.9999, because there
is no margin left for a duplicate to act differently at.

**What this chapter changed.**

It settled that `above_team` was not a separate requirement, and the outfield
list is ten long because of it. That was a fact rather than a judgement, which
is why acting on it did not need an argument.

It also hands the open question about `reliability` a second piece of evidence.
The sensitivity chapter found that `reliability` is the requirement that gives
way first whenever any threshold is tightened. This one finds it correlates 0.7
with `availability`. A requirement that is both fragile and largely redundant is
a requirement with a case to answer, and that one is still open.